In [5]:
# Parámetros
plant_codes  = ""
period_start  = ""     # Si vacío y incremental=True, se calcula por planta como max(Date) en scada_proc - overlap
period_end    = None

# Modo incremental: para cada planta, reprocesa solo desde max(Date) en scada_proc en adelante.
# Si scada_proc aún no tiene datos para la planta, hace proceso completo la primera vez.
# Pon a False para forzar reproceso completo (period_start sigue siendo override).
incremental   = False
# Días de "overlap": cuántos días previos al max(Date) de scada_proc volver a procesar.
# Mínimo 1 recomendado porque c2 agrupa por día completo — si el último día estaba a medias
# en scada_proc, necesita re-procesarse cuando llegue el resto de horas del día.
overlap_days  = 1

In [6]:
def _norm_plant_codes(v):
    """Normaliza plant_codes a lista (una o varias). None/'' = todas.
    Acepta lista/tupla, JSON array o cadena separada por comas."""
    if v is None:
        return []
    if isinstance(v, (list, tuple)):
        return [str(x).strip() for x in v if str(x).strip()]
    s = str(v).strip()
    if not s:
        return []
    try:
        import json as _json
        parsed = _json.loads(s)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except (ValueError, TypeError):
        pass
    return [p.strip() for p in s.split(',') if p.strip()]


import io
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit, current_timestamp

LAKEHOUSE_FILES = '/lakehouse/default/Files'
ABFS_BASE = "abfss://0c4c3bbe-a6ae-404f-b1b7-7052bacfea11@onelake.dfs.fabric.microsoft.com/ae7e1482-8f15-4b1b-8954-8ee500382179"


def _read_sheet(sheet, table=None, header=0):
    """Lee una TABLA ESTRUCTURADA del Excel de config: 1) por NOMBRE de tabla
    (case-insensitive, en cualquier hoja); 2) si no, la (primera) tabla de la hoja
    indicada; 3) ultimo recurso, la hoja completa. Robusto ante renombrados."""
    local = '/tmp/plants_config_proc.xlsx'
    if not Path(local).exists():
        abfs_path = ABFS_BASE + '/Files/Raw PVSyst1/CONFIG/plants_config.xlsx'
        row = spark.read.format('binaryFile').load(abfs_path).first()
        with open(local, 'wb') as _f:
            _f.write(row['content'])
    import openpyxl
    wb = openpyxl.load_workbook(local, data_only=True)
    def _df(ws, ref):
        _d = [[c.value for c in _r] for _r in ws[ref]]
        return pd.DataFrame(_d[1:], columns=_d[0]) if _d else pd.DataFrame()
    if table:
        for ws in wb.worksheets:
            _tm = {k.lower(): k for k in ws.tables}
            if table.lower() in _tm:
                return _df(ws, ws.tables[_tm[table.lower()]].ref)
    if sheet in wb.sheetnames:
        ws = wb[sheet]
        if ws.tables:
            return _df(ws, ws.tables[next(iter(ws.tables))].ref)
    return pd.read_excel(local, sheet_name=sheet, header=header)
def get_active_plants():
    df = _read_sheet('01_Plants', 'T01_plants')
    df = df[df['pr_execution'].astype(str).str.upper().isin(['TRUE','VERDADERO','1','SÍ','SI'])]
    _pl = _norm_plant_codes(plant_codes)
    if _pl:
        df = df[df['plant_code'].isin(_pl)]
    return df.to_dict(orient='records')


def _bool(v, default=False):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return default
    s = str(v).strip().upper()
    return s in ('TRUE', 'VERDADERO', '1', 'SÍ', 'SI', 'YES', 'Y')


def _getf(row, key, default):
    """Lee un float opcional del row, con default si falta o esta vacio."""
    if key not in row.index:
        return default
    v = row[key]
    try:
        return float(v) if v is not None and not pd.isna(v) else default
    except (ValueError, TypeError):
        return default


def _getf_v(v, default):
    """float de un valor suelto, con default si falta o no es numerico."""
    try:
        return float(v) if v is not None and not pd.isna(v) else default
    except (ValueError, TypeError):
        return default


def _monthly_irr_thr(plant_code):
    """07_Irrandiance_Thresholds (tabla T07: plant_code, GHI_m1..POA_m12)."""
    poa, ghi = {}, {}
    try:
        df = _read_sheet('07_Irrandiance_Thresholds', 'T07_irradiance_threshold')
        df.columns = [str(c).strip().upper() for c in df.columns]
        sub = df[df['PLANT_CODE'].astype(str).str.strip() == plant_code]
        if not sub.empty:
            row = sub.iloc[0]
            for m in range(1, 13):
                if f'POA_M{m}' in df.columns:
                    poa[m] = _getf_v(row[f'POA_M{m}'], 0.0)
                if f'GHI_M{m}' in df.columns:
                    ghi[m] = _getf_v(row[f'GHI_M{m}'], 0.0)
    except Exception:
        pass
    return poa, ghi
def get_outlier_bounds():
    """Lee 06_Limits: limites fisicos POR FAMILIA (POA/GHI/DHI/REF/T_AMB).
    Devuelve {familia: {rango_min, rango_max, abrupt_change, dead_threshold}}."""
    try:
        df = _read_sheet('06_Limits', 'T06_limits')
    except Exception:
        df = _read_sheet('04_Outliers')
    fam_pref = [('T_AMB', 'T_AMB'), ('TAMB', 'T_AMB'), ('POA', 'POA'),
                ('GHI', 'GHI'), ('DHI', 'DHI'), ('REF', 'REF')]
    out = {}
    for _, r in df.iterrows():
        param = str(r.get('parametro', '')).strip().upper()
        fam = next((f for pre, f in fam_pref if param.startswith(pre)), None)
        if fam is None:
            continue
        out[fam] = {
            'rango_min':      _getf(r, 'rango_min', None),
            'rango_max':      _getf(r, 'rango_max', None),
            'abrupt_change':  _getf(r, 'abrupt_change', None),
            'dead_threshold': _getf(r, 'dead_threshold', None),
        }
    return out
def _selected_criteria(plant_code):
    """04_Criteria_Selection (tabla T04: plant_code, c1..c9). c1-c7 numerico (-1 off), c8-c9 bool."""
    # CAMBIO CLAVE: Por defecto los criterios están APAGADOS (-1.0 o False) si no se rellenan
    sel = {f'c{i}': (-1.0 if i <= 7 else False) for i in range(1, 10)}
    try:
        sc = _read_sheet('04_Criteria_Selection', 'T04_criteria_selection')
        sc.columns = [str(c).strip().lower() for c in sc.columns]
        sub = sc[sc['plant_code'].astype(str).str.strip() == plant_code]
        if not sub.empty:
            row = sub.iloc[0]
            for n in range(1, 10):
                ck = f'c{n}'
                if ck in sc.columns:
                    # Si el usuario lo deja en blanco en el Excel, asume -1.0 (OFF) o False (OFF)
                    if n <= 7:
                        sel[ck] = _getf_v(row[ck], -1.0)
                    else:
                        sel[ck] = _bool(row[ck], False)
    except Exception:
        pass
    return sel
    
def _sensor_cleaning(plant_code):
    """Reads 05_Sensor_Outliers per plant and returns, per family, the parameters
    for the 3 outlier methods:
        iqr_k       -> Method 1 (IQR/Tukey k; 0 = off)
        dev_rounds  -> Method 2 (up to 3 relative % rounds; each 0/empty = skipped)
        max_diff    -> Method 3 (absolute deviation from the mean, W/m2; 0 = off)
    Accepts the standardized column names (<FAM>_iqr_k/_dev1/_dev2/_dev3/_maxdiff)
    and the older inconsistent names as a fallback during the config transition."""
    fams  = {'POA': 'POA', 'GHI': 'GHI', 'DHI': 'DHI', 'REF': 'REF', 'T_AMB': 'TAMB'}
    alias = {
        'iqr_k':   ['{p}_iqr_k'],
        'dev1':    ['{p}_dev1', '{p}_max_dev_1', '{p}_max_dev', '{p}_max_dev4'],
        'dev2':    ['{p}_dev2', '{p}_max_dev_2', '{p}_max_dev2'],
        'dev3':    ['{p}_dev3', '{p}_max_dev_3', '{p}_max_dev3'],
        'maxdiff': ['{p}_maxdiff', '{p}_max_diff'],
    }
    srow = None
    try:
        so  = _read_sheet('05_Sensor_Outliers', 'T05_sensor_outliers')
        sub = so[so['plant_code'] == plant_code]
        if not sub.empty:
            srow = sub.iloc[0]
    except Exception:
        srow = None

    def _pick(pre, key):
        if srow is None:
            return 0.0
        for name in alias[key]:
            col = name.format(p=pre)
            if col in srow.index:
                return _getf(srow, col, 0.0)
        return 0.0

    out = {}
    for fam, pre in fams.items():
        out[fam] = {
            'iqr_k':      _pick(pre, 'iqr_k'),
            'dev_rounds': [_pick(pre, 'dev1'), _pick(pre, 'dev2'), _pick(pre, 'dev3')],
            'max_diff':   _pick(pre, 'maxdiff'),
        }
    return out


def _albedo_mapping(plant_code):
    """09_Albedo_Mapping: for each albedo measure, which direct (denominator) and
    reflected (numerator) sensors compute it. COMMON is the base; the plant's rows
    add or override by albedo_measure (same convention as 02_Sensors_Mapping).
    Returns an ordered dict {measure: (direct_sensor, reflected_sensor)}."""
    try:
        m = _read_sheet('09_Albedo_Mapping', 'Tabla9')
    except Exception:
        return {}
    m.columns = [str(c).strip().lower() for c in m.columns]
    if 'plant_code' not in m.columns:
        return {}
    def _cell(row, *names):
        for n in names:
            if n in row.index and pd.notna(row[n]):
                return str(row[n]).strip()
        return None
    out = {}
    for pc in ('COMMON', plant_code):
        for _, r in m[m['plant_code'].astype(str).str.strip() == pc].iterrows():
            meas   = _cell(r, 'albedo_measure')
            direct = _cell(r, 'direct_sensor')
            refl   = _cell(r, 'reflected_sensor', 'reflected_sentor')
            if meas and direct and refl:
                out[meas] = (direct, refl)
    return out


def get_criteria(plant_code):
    """Per-plant config for the SCADA processing pipeline, from
    05_Sensor_Outliers + 06_Limits + 04_Criteria_Selection + 07_Irrandiance_Thresholds
    + 09_Albedo_Mapping. EPI / degradation / availability factors are NOT used here
    (they live in 03_EPI_Factors and are consumed by pr_runner / traceability)."""
    _sensor_clean = _sensor_cleaning(plant_code)
    _poa_m, _ghi_m = _monthly_irr_thr(plant_code)

    return {
        # --- Sensor cleaning (05_Sensor_Outliers, per plant and family) ---
        'sensor_clean':              _sensor_clean,
        'sensor_iqr_k':              {fam: p['iqr_k'] for fam, p in _sensor_clean.items()},
        'iqr_daytime_min_wm2':       20.0,

        # --- Active criteria per plant (04_Criteria_Selection) ---
        'selected_criteria':         _selected_criteria(plant_code),

        # --- Monthly irradiance thresholds for c5/c6 (07_Irrandiance_Thresholds) ---
        'monthly_thr_poa':           _poa_m,
        'monthly_thr_ghi':           _ghi_m,

        # --- Physical per-family limits for anomaly detection (06_Limits) ---
        'outlier_bounds':            get_outlier_bounds(),

        # --- Albedo per 09_Albedo_Mapping (measure -> direct/reflected sensors) ---
        'albedo_mapping':            _albedo_mapping(plant_code),
    }

In [7]:
def _suffix(name):
    return name.split('_')[-1] if '_' in name else ''


# =============================================================================
#  Anomaly detection — flagging por sensor antes de cualquier criterio.
#  Tipos:
#    - out_of_range : valor fuera de [min_bound, max_bound]
#    - missing      : NaN
#    - dead         : derivada < dead_thr mientras |valor| > min_significant
#    - abrupt       : |derivada| > abrupt_thr
#  Para cada sensor <col> creamos:
#    - <col>          : valor numerico (NaN si es anomalia)
#    - <col>_flag     : string 'ok' / 'out_of_range' / 'missing' / 'dead' / 'abrupt'
# =============================================================================

def detect_anomalies_for(df, col, min_bound, max_bound, dead_thr, abrupt_thr, min_significant=5.0):
    """Marca anomalias en una columna. Devuelve (series_limpia, series_flag)."""
    v = pd.to_numeric(df[col], errors='coerce')
    flag = pd.Series('ok', index=df.index, dtype=object)

    flag[v.isna()] = 'missing'
    mask_oor = (~v.isna()) & ((v < min_bound) | (v > max_bound))
    flag[mask_oor] = 'out_of_range'

    diff = v.diff().abs()
    mask_dead = (~v.isna()) & (v.abs() > min_significant) & (diff < dead_thr) & (flag == 'ok')
    flag[mask_dead] = 'dead'
    mask_abrupt = (~v.isna()) & (diff > abrupt_thr) & (flag == 'ok')
    flag[mask_abrupt] = 'abrupt'

    # Solo se anula el valor para 'missing' y 'out_of_range' (sin dato usable).
    # 'dead' y 'abrupt' son fallos de sensor pero el valor se CONSERVA: se marca
    # la flag (control interno) y se sigue calculando con el dato real.
    clean = v.where(~flag.isin(['missing', 'out_of_range']), np.nan)
    return clean, flag


def apply_anomaly_detection(df, crit):
    """Aplica detect_anomalies_for a cada sensor usando los limites POR FAMILIA
    de 06_Limits (crit['outlier_bounds'])."""
    bounds = crit.get('outlier_bounds', {})
    fam_pref = [('GHI_', 'GHI'), ('POA_', 'POA'), ('REF_', 'REF'), ('DHI_', 'DHI')]
    irrad_cols = [c for c in df.columns
                  if any(c.upper().startswith(p) for p, _ in fam_pref)
                  and not c.endswith('_filt') and not c.endswith('_flag')]
    for c in irrad_cols:
        fam = next((f for p, f in fam_pref if c.upper().startswith(p)), None)
        b = bounds.get(fam, {})
        clean, flag = detect_anomalies_for(
            df, c,
            min_bound=b.get('rango_min', -6.0) if b.get('rango_min') is not None else -6.0,
            max_bound=b.get('rango_max', 1500.0) if b.get('rango_max') is not None else 1500.0,
            dead_thr=b.get('dead_threshold', 0.0001) if b.get('dead_threshold') is not None else 0.0001,
            abrupt_thr=b.get('abrupt_change', 800.0) if b.get('abrupt_change') is not None else 800.0,
        )
        df[c] = clean
        df[f'{c}_flag'] = flag

    bt = bounds.get('T_AMB', {})
    tamb_cols = [c for c in df.columns
                 if c.upper().startswith('T_AMB') and not c.endswith('_filt') and not c.endswith('_flag')]
    for c in tamb_cols:
        clean, flag = detect_anomalies_for(
            df, c,
            min_bound=bt.get('rango_min', -30.0) if bt.get('rango_min') is not None else -30.0,
            max_bound=bt.get('rango_max', 50.0) if bt.get('rango_max') is not None else 50.0,
            dead_thr=bt.get('dead_threshold', 0.0001) if bt.get('dead_threshold') is not None else 0.0001,
            abrupt_thr=bt.get('abrupt_change', 4.0) if bt.get('abrupt_change') is not None else 4.0,
            min_significant=1.0,
        )
        df[c] = clean
        df[f'{c}_flag'] = flag

    return df
# =============================================================================
#  Outlier IQR (Tukey) por sensor — sobre valores diurnos de todo el periodo.
#  Marca como outlier (NaN + flag 'outlier_iqr') lo que cae fuera de
#  [Q1 - k*IQR, Q3 + k*IQR]. Aplica a GHI/DHI/REF/POA.
# =============================================================================

def apply_iqr_outliers(df, k_map, daytime_min=20.0):
    # k_map: dict {familia: k} (0 = off). Familias: POA/GHI/DHI/REF.
    fam_of = {'POA_': 'POA', 'GHI_': 'GHI', 'DHI_': 'DHI', 'REF_': 'REF'}
    cols = [c for c in df.columns
            if any(c.upper().startswith(f) for f in fam_of)
            and not c.endswith('_filt') and not c.endswith('_flag')]
    for c in cols:
        _fam = next((fam_of[f] for f in fam_of if c.upper().startswith(f)), None)
        k = (k_map.get(_fam, 0.0) if isinstance(k_map, dict) else (k_map or 0.0))
        if not k or k <= 0:
            continue
        s   = pd.to_numeric(df[c], errors='coerce')
        day = s >= daytime_min                      # solo diurno (evita el sesgo de ceros nocturnos)
        dv  = s[day]
        if dv.notna().sum() < 8:
            continue
        q1, q3 = dv.quantile(0.25), dv.quantile(0.75)
        iqr = q3 - q1
        if iqr <= 0:
            continue
        lo, hi = q1 - k * iqr, q3 + k * iqr
        out = day & ((s < lo) | (s > hi))
        if not bool(out.any()):
            continue
        df.loc[out, c] = np.nan                     # outlier IQR SI se descarta
        fcol = f'{c}_flag'
        if fcol in df.columns:
            df.loc[out & (df[fcol] == 'ok'), fcol] = 'outlier_iqr'
    return df


# =============================================================================
#  Cross-sensor cleaning per family (Methods 2 and 3, applied cumulatively on
#  the survivors of the previous step).
#  Method 2 - up to 3 rounds of RELATIVE (%) deviation from the median: each round
#    computes the median of the still-active sensors and drops every sensor
#    deviating more than that round's threshold, then recomputes the mean for the
#    next round. Rounds with threshold 0/empty are skipped.
#  Method 3 - ABSOLUTE deviation from the median (W/m2): drops sensors whose
#    |value - median| exceeds max_diff. Skipped if 0/empty.
#  Needs >= 3 sensors in the family to compare; otherwise it is a passthrough.
#  (Method 1, IQR/Tukey, runs earlier per sensor in apply_iqr_outliers and has
#  already NaN-ed its outliers on the raw columns read here.)
# =============================================================================

def _cross_sensor_filter(df, cols, dev_rounds, max_diff, min_sensors=3):
    if not cols:
        return []
    raw    = df[cols].astype(float)
    active = raw.notna()
    if len(cols) >= min_sensors:
        for p in dev_rounds:                        # Method 2: relative % rounds
            if not p or p <= 0:
                continue
            masked   = raw.where(active)
            center   = masked.median(axis=1)
            n_active = active.sum(axis=1)
            with np.errstate(divide='ignore', invalid='ignore'):
                reldev = masked.sub(center, axis=0).abs().div(center.abs().replace(0, np.nan), axis=0)
            rows = n_active >= min_sensors
            for c in cols:
                deact = rows & (reldev[c] > p)
                if bool(deact.any()):
                    active.loc[deact, c] = False
        if max_diff and max_diff > 0:               # Method 3: absolute deviation
            masked   = raw.where(active)
            center   = masked.median(axis=1)
            n_active = active.sum(axis=1)
            absdev   = masked.sub(center, axis=0).abs()
            rows = n_active >= min_sensors
            for c in cols:
                deact = rows & (absdev[c] > max_diff)
                if bool(deact.any()):
                    active.loc[deact, c] = False
    for c in cols:
        df[f'{c}_filt'] = raw[c].where(active[c])
    return [f'{c}_filt' for c in cols]


def _family_cols(df, prefix):
    return sorted([c for c in df.columns
                   if c.upper().startswith(prefix)
                   and not c.endswith('_filt') and not c.endswith('_flag')])


def _apply_family_cleaning(df, prefix, avg_col, params):
    cols = _family_cols(df, prefix)
    filt_cols = _cross_sensor_filter(df, cols,
                                     params.get('dev_rounds', []),
                                     params.get('max_diff', 0.0))
    df[avg_col] = (df[filt_cols].astype(float).mean(axis=1, skipna=True)
                   if cols else np.nan)
    return df


def filter_poa_sensors(df, params):
    return _apply_family_cleaning(df, 'POA_', 'POA_avg_filt', params)


def filter_ghi_sensors(df, params):
    return _apply_family_cleaning(df, 'GHI_', 'GHI_avg_filt', params)


def filter_dhi_sensors(df, params):
    return _apply_family_cleaning(df, 'DHI_', 'DHI_avg_filt', params)


def filter_tamb_sensors(df, params):
    return _apply_family_cleaning(df, 'T_AMB', 'T_AMB_avg_filt', params)


def filter_ref_sensors(df, params):
    return _apply_family_cleaning(df, 'REF_', 'REF_avg_filt', params)


# =============================================================================
#  Albedo — computed per 09_Albedo_Mapping (reflected / direct)
# =============================================================================

def compute_albedo(df, mapping):
    """Albedo per 09_Albedo_Mapping. For each configured measure:
        ALB_measure = reflected_sensor / direct_sensor
    Convention (same as the Plaza II Excel):
        direct NaN or reflected NaN -> NaN (empty)
        direct == 0 or ratio < 0    -> 0
    Only the measures listed in the mapping whose sensors exist in df are created.
    ALB_avg_filt = mean of the created ALB columns within [0, 1)."""
    alb_cols = []
    for meas, (direct, refl) in mapping.items():
        if direct not in df.columns or refl not in df.columns:
            continue
        den_v = df[direct].astype(float)
        ref_v = df[refl].astype(float)
        valid_inputs = (~den_v.isna()) & (~ref_v.isna())
        with np.errstate(divide='ignore', invalid='ignore'):
            ratio = ref_v / den_v.replace(0, np.nan)   # den==0 -> NaN
        alb = ratio.where(ratio >= 0, 0.0)             # den==0 or ratio<0 -> 0
        df[meas] = alb.where(valid_inputs, np.nan)     # non-numeric inputs -> NaN
        alb_cols.append(meas)

    if alb_cols:
        masked = df[alb_cols].where((df[alb_cols] > 0) & (df[alb_cols] < 1))
        df['ALB_avg_filt'] = masked.mean(axis=1, skipna=True)
    else:
        df['ALB_avg_filt'] = np.nan
    return df


# =============================================================================
#  4 Criterios operativos (estilo Plaza II)
# =============================================================================

def _seasonal_poa_threshold(date_series, crit):
    """Threshold POA segun mes (Apr-Sep / Oct-Mar)."""
    month = date_series.dt.month
    return np.where(
        (month >= 4) & (month <= 9),
        crit['poa_threshold_summer_wm2'],
        crit['poa_threshold_winter_wm2']
    )


def compute_criteria(df, crit):
    """9 criterios (IEC 61724-2). Evaluados solo si están activos."""
    sel = crit.get('selected_criteria', {})

    # Umbral de irradiancia MENSUAL por familia (07_Irrandiance_Thresholds)
    poa_m = crit.get('monthly_thr_poa', {})
    ghi_m = crit.get('monthly_thr_ghi', {})
    month = df['Date'].dt.month
    poa_thr = month.map(lambda m: poa_m.get(int(m), 0.0)).astype(float)
    ghi_thr = month.map(lambda m: ghi_m.get(int(m), 0.0)).astype(float)
    df['poa_above_thr'] = (df['POA_avg_filt'].fillna(0).values >= poa_thr.values).astype(int)
    df['ghi_above_thr'] = (df['GHI_avg_filt'].fillna(0).values >= ghi_thr.values).astype(int)

    diffs = df['Date'].diff().dropna()
    dt_h = float(diffs.mode().iloc[0].total_seconds() / 3600) if not diffs.empty else 0.25

    # Funciones auxiliares para saber si un criterio está vivo
    def _is_active(ckey):
        v = sel.get(ckey, -1.0)
        return v is not None and v >= 0
        
    def _get_thr(ckey):
        v = sel.get(ckey, -1.0)
        return v if (v is not None and v >= 0) else -1.0

    # C1-C4: salud del avg_filt.
    def _avg_crit(col, ckey):
        if not _is_active(ckey):
            return pd.Series(1, index=df.index)  # OFF -> siempre válido (1)
        thr_val = _get_thr(ckey)
        s = df[col]
        if thr_val > 0:
            return (s.fillna(-1.0) >= thr_val).astype(int)
        return s.notna().astype(int)

    df['c1'] = _avg_crit('POA_avg_filt', 'c1')
    df['c2'] = _avg_crit('GHI_avg_filt', 'c2')
    df['c3'] = _avg_crit('DHI_avg_filt', 'c3')
    df['c4'] = _avg_crit('REF_avg_filt', 'c4')

    df['_day'] = df['Date'].dt.date
    
    # C5: POA
    dh_poa = df.groupby('_day')['poa_above_thr'].transform('sum') * dt_h
    df['daily_hours_above'] = dh_poa
    if _is_active('c5'):
        df['c5'] = (dh_poa >= _get_thr('c5')).astype(int)
    else:
        df['c5'] = 1

    # C6: GHI
    dh_ghi = df.groupby('_day')['ghi_above_thr'].transform('sum') * dt_h
    df['daily_hours_above_ghi'] = dh_ghi
    if _is_active('c6'):
        df['c6'] = (dh_ghi >= _get_thr('c6')).astype(int)
    else:
        df['c6'] = 1

    # C7: setpoint
    sp_col = next((c for c in df.columns if c.upper().startswith('SETPOINT')), None)
    sp = df[sp_col].astype(float).fillna(0) if sp_col else pd.Series(0.0, index=df.index)
    if _is_active('c7'):
        df['c7'] = (sp >= _get_thr('c7')).astype(int)
    else:
        df['c7'] = 1

    # C8: WA
    wa_col = next((c for c in df.columns
                   if (c.upper().startswith('WA_') or c.upper() == 'WA')
                   and not c.endswith('_flag') and not c.endswith('_filt')), None)
    wa = df[wa_col].astype(float).fillna(0) if wa_col else pd.Series(0.0, index=df.index)
    if sel.get('c8', False):
        df['c8'] = (wa != 1).astype(int)
    else:
        df['c8'] = 1

    # C9: PF. El criterio C9 usa SIEMPRE la columna 'PF' (destino del mapping PF_01->PF).
    if sel.get('c9', False):
        if 'PF' in df.columns:
            pf = df['PF'].astype(float)
            df['c9'] = ((pf >= 0.98) & (pf <= 1.0)).astype(int)
        else:
            df['c9'] = 0  # Criterio encendido pero falta la columna PF = Falla
    else:
        df['c9'] = 1
    # Validez global = producto estricto de todos (los inactivos ya son 1)
    av = df['c1'] * df['c2'] * df['c3'] * df['c4'] * df['c5'] * df['c6'] * df['c7'] * df['c8'] * df['c9']
    df['all_valid'] = av.astype(int)

    df['hours_aggregated'] = np.where(df['all_valid'] == 1, dt_h, 0.0)

    return df.drop(columns=['_day'])

In [8]:
from datetime import timedelta

spark.conf.set('spark.databricks.delta.schema.autoMerge.enabled', 'true')


def get_last_proc_date(plant_code):
    if not spark.catalog.tableExists('scada_proc'):
        return None
    row = spark.sql(f"""
        SELECT MAX(Date) AS mx
        FROM scada_proc
        WHERE plant_code = '{plant_code}'
    """).first()
    return row['mx']


def process_plant(plant_code, crit):
    eff_start = period_start
    if eff_start is None and incremental:
        last_date = get_last_proc_date(plant_code)
        if last_date is not None:
            floor_day = pd.Timestamp(last_date).normalize()
            eff_start = (floor_day - timedelta(days=overlap_days)).isoformat()
            print(f"  incremental: reproceso desde {eff_start}  (último proc {last_date}, overlap {overlap_days}d)")
        else:
            print(f"  primera ejecución de {plant_code}: proceso completo")
    elif eff_start is None:
        print(f"  incremental=False: proceso completo")

    where = [f"plant_code = '{plant_code}'"]
    if eff_start:
        where.append(f"Date >= TIMESTAMP'{eff_start}'")
    if period_end:
        where.append(f"Date <= TIMESTAMP'{period_end}'")
    sql = f"SELECT * FROM scada_raw WHERE {' AND '.join(where)} ORDER BY Date"

    sdf = spark.sql(sql)
    df = sdf.toPandas()
    if df.empty:
        return 0
    df = df.drop(columns=['source_file', 'ingested_at'], errors='ignore')
    df = df.sort_values('Date').reset_index(drop=True)

    # ---- Pipeline ----------------------------------------------------------
    # 1) Anomaly detection per sensor (out_of_range / missing / dead / abrupt)
    df = apply_anomaly_detection(df, crit)

    # 1b) Method 1: IQR/Tukey per sensor (daytime) -> discards values outside the fence
    df = apply_iqr_outliers(df, crit['sensor_iqr_k'], crit['iqr_daytime_min_wm2'])

    # 2) Cross-sensor cleaning per family: Method 2 (% rounds) + Method 3 (absolute).
    df = filter_poa_sensors(df, crit['sensor_clean']['POA'])
    df = filter_ghi_sensors(df, crit['sensor_clean']['GHI'])
    df = filter_dhi_sensors(df, crit['sensor_clean']['DHI'])
    df = filter_ref_sensors(df, crit['sensor_clean']['REF'])
    df = filter_tamb_sensors(df, crit['sensor_clean']['T_AMB'])

    # 3) Albedo per 09_Albedo_Mapping (reflected / direct sensors)
    df = compute_albedo(df, crit['albedo_mapping'])

    # 4) Criterios operativos C1-C4
    df = compute_criteria(df, crit)

    df['processed_at'] = datetime.utcnow()
    sdf_out = spark.createDataFrame(df)

    table = 'scada_proc'
    if spark.catalog.tableExists(table):
        target = DeltaTable.forName(spark, table)
        # Mapear solo columnas del source (cada planta tiene sensores distintos).
        _set = {col: f"s.`{col}`" for col in sdf_out.columns}
        (target.alias('t')
            .merge(sdf_out.alias('s'),
                   "t.plant_code = s.plant_code AND t.Date = s.Date")
            .whenMatchedUpdate(set=_set)
            .whenNotMatchedInsert(values=_set)
            .execute())
    else:
        (sdf_out.write
            .format('delta')
            .option('mergeSchema', 'true')
            .partitionBy('plant_code')
            .mode('append')
            .saveAsTable(table))

    return len(df)



def get_all_dest_columns():
    """Universo COMPLETO de columnas destino del mapping (todas las plantas)."""
    m = _read_sheet('02_Sensors_Mapping', 'T02_sensor_mapping')
    dests = []
    for _, r in m.iterrows():
        d = str(r['columna_destino']).strip() if pd.notna(r['columna_destino']) else None
        if d and d not in dests:
            dests.append(d)
    return [d for d in dests if d != 'Date']


def ensure_full_schema(table):
    """Garantiza que la tabla tenga TODAS las columnas de sensores del mapping
    (las ausentes se anaden como DOUBLE null). Asi scada_proc tiene tantas
    columnas de sensores como 02_Sensors_Mapping, sin depender de que planta
    se procese ni del orden."""
    if not spark.catalog.tableExists(table):
        return
    existing = set(spark.table(table).columns)
    for d in get_all_dest_columns():
        if d not in existing:
            spark.sql(f"ALTER TABLE {table} ADD COLUMNS (`{d}` DOUBLE)")
            print(f"  schema: +{d} en {table}")


ensure_full_schema('scada_proc')

results = []
for p in get_active_plants():
    plant = p['plant_code']
    print(f"\n-> {plant}")
    try:
        crit = get_criteria(plant)
        n = process_plant(plant, crit)
        print(f"  {n:,} filas escritas en scada_proc")
        results.append({'plant_code': plant, 'rows': n, 'status': 'ok', 'error': None})
    except Exception as e:
        import traceback as _tb
        err = _tb.format_exc()[:2000]
        print(f"  FAILED: {err}")
        results.append({'plant_code': plant, 'rows': 0, 'status': 'failed', 'error': err})

In [9]:
import json, notebookutils
summary = {
    'plants_processed': len(results),
    'ok':     sum(1 for r in results if r['status'] == 'ok'),
    'failed': sum(1 for r in results if r['status'] == 'failed'),
}
print(json.dumps(summary, indent=2))
try:
    from pathlib import Path as _P
    _P('/lakehouse/default/Files/LOGS').mkdir(parents=True, exist_ok=True)
    _P('/lakehouse/default/Files/LOGS/scada_proc_results.json').write_text(json.dumps(results, indent=2, default=str), encoding='utf-8')
except Exception as _e:
    print('no se pudo volcar results:', _e)
notebookutils.notebook.exit(json.dumps(summary))
